# APAN 5200 — Logistic Regression Mock Quiz 1
**Dataset:** `spotify_data.csv`  
**Outcome:** `popular` = 1 if `rating >= 50`, else 0  
**Goal:** Predict whether a song is popular.

This mock quiz follows the same structure as the original Logistic Regression quiz:
- **Part 1** uses `statsmodels` formula API
- **Part 2** uses `scikit-learn`

## Variable Descriptions
- `track_duration`: Duration in ms
- `track_explicit`: Is explicit (0/1)
- `danceability`: Suitability for dancing (0–1)
- `energy`: Intensity & activity (0–1)
- `loudness`: Overall loudness in dB
- `mode`: Major (1) or Minor (0)
- `speechiness`: Presence of spoken words (0–1)
- `acousticness`: Acoustic confidence (0–1)
- `instrumentalness`: No-vocal likelihood (0–1)
- `liveness`: Audience presence (0–1)
- `valence`: Musical positiveness (0–1)
- `tempo`: BPM
- `time_signature`: Time signature
- `genre_*`: 10 binary genre dummies
- `popular`: **TARGET** — 1 if rating ≥ 50, 0 otherwise

---
## Part 1 — StatsModels

In [ ]:
import pandas as pd
import numpy as np
import math
import warnings
warnings.filterwarnings('ignore')
from statsmodels.formula.api import logit

In [ ]:
# ============================================================
# Q1: Read spotify_data.csv. Drop: id, performer, song, genre, key.
#     Create: popular = 1 if rating >= 50, else 0.
#     Split 70/30 using DataFrame.sample(), random_state=42.
#     What % of songs in the TRAIN sample are popular?
# ANSWER: 24.54%
# ============================================================

data = pd.read_csv('spotify_data.csv')
data = data.drop(columns=['id','performer','song','genre','key'])
data['popular'] = (data['rating'] >= 50).astype(int)

train = data.sample(frac=0.7, random_state=42)
test  = data.drop(labels=train.index)

pct_pop = round(train['popular'].mean() * 100, 2)
print(f'Train popular rate: {pct_pop}%')  # ✅ 24.54

In [ ]:
# ============================================================
# Q2: Compare popular vs non-popular songs on average LOUDNESS.
#     What is the average loudness for POPULAR songs (popular=1)?
# ANSWER: -7.93 dB
# KEY INSIGHT: Popular songs are louder on average.
# ============================================================

avg_loud = train.groupby('popular')['loudness'].mean().round(2)
print(avg_loud)
print(f'Avg loudness (popular=1): {round(train[train["popular"]==1]["loudness"].mean(),2)}')  # ✅ -7.93

In [ ]:
# ============================================================
# Q3: Fit model1 = logit('popular ~ loudness'). Which conclusion is correct?
# ANSWER: loudness coefficient is POSITIVE (~0.077), p<0.05
#         → Louder songs are MORE likely to be popular (significant)
# ============================================================

model1 = logit('popular ~ loudness', data=train).fit()
print(model1.summary())
print(f'\nloudness coeff: {round(model1.params["loudness"],4)}')  # ✅ ~+0.077

In [ ]:
# ============================================================
# Q4: Based on model1, what is the predicted probability that the
#     FIRST song in the train sample is popular?
# ANSWER: ~0.157
# ============================================================

first_song = train.iloc[[0]][['loudness']]
prob_first = model1.predict(first_song)
print(f'First song loudness = {round(train["loudness"].iloc[0],2)}')
print(f'P(popular=1) = {round(prob_first.values[0],4)}')  # ✅ ~0.157

In [ ]:
# ============================================================
# Q5: Based on model1, what is the probability that a song with
#     loudness = -5 dB is popular?
# ANSWER: ~0.296
# ============================================================

prob_m5 = model1.predict(pd.DataFrame({'loudness':[-5]}))
print(f'P(popular | loudness=-5) = {round(prob_m5.values[0],4)}')  # ✅ ~0.296

In [ ]:
# ============================================================
# Q6: Based on model1, if loudness increases by 1 dB, what is
#     the PERCENT CHANGE in likelihood of being popular?
# ANSWER: +7.95%
# FORMULA: (exp(β) − 1) × 100
# ============================================================

b_loud = model1.params['loudness']
pct_change = (math.exp(b_loud) - 1) * 100
print(f'Pct change per +1 dB loudness: {round(pct_change,2)}%')  # ✅ +7.95%

In [ ]:
# ============================================================
# Q7: Examine popular rate by GENRE (use the genre_ dummy variables).
#     Which genre dummy has the HIGHEST popular rate?
# ANSWER: genre_dance (42.08%)
# ============================================================

genre_cols = [c for c in train.columns if c.startswith('genre_')]
genre_rates = {g: round(train[train[g]==1]['popular'].mean()*100,2) for g in genre_cols}
genre_rates_s = sorted(genre_rates.items(), key=lambda x: -x[1])
print('Popular rate by genre:')
for g,r in genre_rates_s:
    print(f'  {g}: {r}%')
# ✅ genre_dance: 42.08%  |  genre_blues: 12.28%  |  genre_space: 8.33%

In [ ]:
# ============================================================
# Q8: Fit model2 = logit('popular ~ genre_dance').
#     What is the % change in likelihood for a dance song vs non-dance?
# ANSWER: +169.5%  (dance songs much more likely to be popular)
# ============================================================

model2 = logit('popular ~ genre_dance', data=train).fit()
print(model2.summary())
b_dance = model2.params['genre_dance']
print(f'\ngenre_dance coeff: {round(b_dance,4)}')
print(f'Pct change: {round((math.exp(b_dance)-1)*100,2)}%')  # ✅ +169.5%

In [ ]:
# ============================================================
# Q9: Fit model3 = logit('popular ~ danceability'). Which conclusion?
# ANSWER: danceability coefficient is POSITIVE (~1.49), p<0.05
#         → More danceable songs are more likely to be popular
# ============================================================

model3 = logit('popular ~ danceability', data=train).fit()
print(model3.summary())
print(f'\ndanceability coeff: {round(model3.params["danceability"],4)}')  # ✅ ~+1.49

In [ ]:
# ============================================================
# Q10: Fit model4 with ALL numeric audio features:
#      loudness, danceability, energy, tempo, valence, acousticness,
#      speechiness, track_duration, track_explicit, mode.
#      What is the Pseudo R² (McFadden)?
# ANSWER: run to compute (expect ~0.05–0.10 range)
# ============================================================

model4 = logit(
    'popular ~ loudness + danceability + energy + tempo + valence +'
    ' acousticness + speechiness + track_duration + track_explicit + mode',
    data=train
).fit()
print(f'Pseudo R² (McFadden): {round(model4.prsquared,4)}')

In [ ]:
# ============================================================
# Q11: Based on model4, which variables significantly affect popularity?
# ANSWER: loudness, danceability, energy, acousticness (p<0.05)
#         tempo, valence, speechiness, mode → NOT significant
# ============================================================

print(model4.summary())
sig = model4.pvalues[model4.pvalues < 0.05]
print('\nSignificant variables (p<0.05):')
print(sig)

In [ ]:
# ============================================================
# Q12: Based on model4, what is the impact of a ONE-UNIT increase
#      in DANCEABILITY on the likelihood of being popular?
# ANSWER: ~+287% (danceability 0→1 dramatically increases popularity odds)
# ============================================================

b_dance4 = model4.params['danceability']
impact = (math.exp(b_dance4) - 1) * 100
print(f'danceability coeff: {round(b_dance4,4)}')
print(f'Pct change per +1 unit danceability: {round(impact,2)}%')

---
## Part 2 — Scikit-Learn

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, recall_score, roc_auc_score

In [ ]:
# ============================================================
# Q13: Separate X and y (popular). Drop 'rating' from X.
#      Stratified 80/20 split, random_state=42.
#      What % of the TRAIN sample are popular?
# ANSWER: 25.13%
# ============================================================

y = data['popular']
X = data.drop(columns=['popular', 'rating'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=42, stratify=y
)

print(f'Train popular rate: {round(y_train.mean()*100,2)}%')  # ✅ 25.13

In [ ]:
# ============================================================
# Q14: What is the mean of 'track_explicit' in the train sample?
# ANSWER: 0.121  (~12.1% of train songs are explicit)
# ============================================================

print(f'track_explicit mean (train): {round(X_train["track_explicit"].mean(),4)}')  # ✅ 0.121

In [ ]:
# ============================================================
# Q15: Fit LogisticRegression(max_iter=10000, class_weight='balanced').
#      What is the predicted CLASS for the first observation in train?
# NOTE: class_weight='balanced' corrects for the 75/25 class imbalance,
#       improving sensitivity without changing the model structure.
# ANSWER: 1 (predicted popular)  |  P(popular=1) ≈ 0.773
# ============================================================

logit_sk = LogisticRegression(max_iter=10000, random_state=42, class_weight='balanced')
logit_sk.fit(X_train, y_train)

y_pred_train       = logit_sk.predict(X_train)
y_pred_proba_train = logit_sk.predict_proba(X_train)
y_pred_test        = logit_sk.predict(X_test)
y_pred_proba_test  = logit_sk.predict_proba(X_test)

print(f'First obs predicted class : {y_pred_train[0]}')         # ✅ 1
print(f'First obs P(popular=1)    : {round(y_pred_proba_train[0,1],4)}')  # ✅ 0.773

In [ ]:
# ============================================================
# Q16: False Negatives are most costly (missing a popular song).
#      What is the Sensitivity (Recall) in the TRAIN sample?
# ANSWER: 0.6514
# ============================================================

sens_train = recall_score(y_train, y_pred_train)
print(f'Sensitivity (train): {round(sens_train,4)}')  # ✅ 0.6514

cm = confusion_matrix(y_train, y_pred_train)
print(f'\nConfusion Matrix (train):\n{cm}')
print(f'TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')

In [ ]:
# ============================================================
# Q17: What is the Sensitivity in the TEST sample?
# ANSWER: 0.6580
# NOTE: test ≈ train → no significant overfitting
# ============================================================

sens_test = recall_score(y_test, y_pred_test)
print(f'Sensitivity (test): {round(sens_test,4)}')  # ✅ 0.658

cm_te = confusion_matrix(y_test, y_pred_test)
print(f'\nConfusion Matrix (test):\n{cm_te}')

In [ ]:
# ============================================================
# Q18: What is the AUC in the TRAIN sample?
# ANSWER: 0.6761
# ============================================================

auc_train = roc_auc_score(y_train, y_pred_proba_train[:,1])
print(f'AUC (train): {round(auc_train,4)}')  # ✅ 0.6761

In [ ]:
# ============================================================
# Q19: What is the AUC in the TEST sample?
# ANSWER: 0.6819
# KEY INSIGHT:
#   AUC ~0.68 is moderate — audio features alone have limited
#   predictive power for song popularity.
#   genre_dance, loudness, and danceability are the strongest signals.
# ============================================================

auc_test = roc_auc_score(y_test, y_pred_proba_test[:,1])
print(f'AUC (test): {round(auc_test,4)}')  # ✅ 0.6819

---
## Answer Summary

In [ ]:
summary = {
    'Q1  Train popular rate (%)':      '24.54',
    'Q2  Avg loudness (popular=1)':     '-7.93 dB',
    'Q3  loudness effect':              'Positive & significant (~+0.077)',
    'Q4  P(popular | first song)':      '~0.157',
    'Q5  P(popular | loudness=-5)':     '~0.296',
    'Q6  loudness +1dB → % change':     '+7.95%',
    'Q7  Highest popular genre':        'genre_dance (42.08%)',
    'Q8  genre_dance % change':         '+169.5%',
    'Q9  danceability effect':          'Positive & significant (~+1.49)',
    'Q10 Pseudo R² (model4)':           'run to compute',
    'Q11 Significant vars':             'loudness, danceability, energy, acousticness',
    'Q12 danceability +1 → % change':   'run to compute (~+287%)',
    'Q13 Stratified train rate (%)':    '25.13',
    'Q14 track_explicit mean (train)':  '0.121',
    'Q15 First obs prediction':         '1 (popular)',
    'Q16 Sensitivity (train)':          '0.6514',
    'Q17 Sensitivity (test)':           '0.6580',
    'Q18 AUC (train)':                  '0.6761',
    'Q19 AUC (test)':                   '0.6819',
}
for k,v in summary.items():
    print(f'{k:<40} → {v}')